# LAB 03 — Word Representations and Embeddings
Co-occurrence → PPMI → Word2Vec (skip-gram / CBOW) → similarity, analogy, semantic search.
Chạy từ trên xuống. Corpus: `../lab02/data/c4-train.00000-of-01024-30K.json.gz` (dùng 8,000 documents đầu của C4 cho nhanh).

In [1]:
import gzip, json, re, time, random, csv
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from gensim.models import Word2Vec
from cooccurrence import (tokenize, build_vocabulary, build_cooccurrence_matrix,
                          cosine_similarity, most_similar, ppmi)
random.seed(0); np.random.seed(0)
RESULTS = []   # (experiment, config, metric, value)
def log(exp, cfg, metric, val): RESULTS.append((exp, cfg, metric, val))

docs = [json.loads(l)["text"] for l in gzip.open("../lab02/data/c4-train.00000-of-01024-30K.json.gz","rt",encoding="utf8")][:8000]
# mỗi câu là một "context unit": window không vượt qua ranh giới câu
sentences = [tokenize(s) for d in docs for s in re.split(r"[.!?\n]+", d.lower())]
sentences = [s for s in sentences if len(s) >= 3]
print(len(docs), "docs;", len(sentences), "sentences;", sum(map(len, sentences)), "tokens")

8000 docs; 165900 sentences; 2875396 tokens


## Experiment 1 — Word-context matrix
Trước hết kiểm tra trên corpus nhỏ trong đề (window = 1), rồi chạy trên C4 với window 1/2/5.

In [2]:
toy = [s.split() for s in ["the cat eats fish","the cat likes milk","the dog eats meat","the dog likes fish"]]
tv, tw2i = build_vocabulary(toy)
Xt = build_cooccurrence_matrix(toy, tw2i, window=1)
vocab_order = ["cat","dog","eats","likes","fish","milk","meat"]
print(pd.DataFrame(Xt.toarray()[[tw2i[w] for w in vocab_order]][:, [tw2i[w] for w in vocab_order]],
                   index=vocab_order, columns=vocab_order))
print("cos(cat,dog) =", cosine_similarity(Xt[tw2i["cat"]], Xt[tw2i["dog"]]))

       cat  dog  eats  likes  fish  milk  meat
cat    0.0  0.0   1.0    1.0   0.0   0.0   0.0
dog    0.0  0.0   1.0    1.0   0.0   0.0   0.0
eats   1.0  1.0   0.0    0.0   1.0   0.0   1.0
likes  1.0  1.0   0.0    0.0   1.0   1.0   0.0
fish   0.0  0.0   1.0    1.0   0.0   0.0   0.0
milk   0.0  0.0   0.0    1.0   0.0   0.0   0.0
meat   0.0  0.0   1.0    0.0   0.0   0.0   0.0
cos(cat,dog) = 0.9999999403953552


In [3]:
vocab, w2i = build_vocabulary(sentences, min_count=5, max_size=10000)
print("vocab size:", len(vocab))
rows, mats = [], {}
for w in (1, 2, 5):
    t = time.time()
    X = build_cooccurrence_matrix(sentences, w2i, window=w); mats[w] = X
    P = ppmi(X)
    rows.append(dict(window=w, vocab=len(vocab), matrix=f"{X.shape[0]}x{X.shape[1]}", nonzero=X.nnz,
                     density=f"{X.nnz/X.shape[0]/X.shape[1]:.2%}", build_s=round(time.time()-t,1),
                     cos_doctor_physician_count=round(cosine_similarity(X[w2i['doctor']],X[w2i['physician']]),3),
                     cos_doctor_physician_ppmi=round(cosine_similarity(P[w2i['doctor']],P[w2i['physician']]),3),
                     cos_doctor_banana_ppmi=round(cosine_similarity(P[w2i['doctor']],P[w2i['banana']]),3),
                     cos_cat_dog_ppmi=round(cosine_similarity(P[w2i['cat']],P[w2i['dog']]),3)))
    for k in ("cos_doctor_physician_count","cos_doctor_physician_ppmi","cos_cat_dog_ppmi"):
        log("exp1_cooc", f"window={w}", k, rows[-1][k])
    log("exp1_cooc", f"window={w}", "nonzero", X.nnz)
pd.DataFrame(rows).set_index("window").T

vocab size: 10000


window,1,2,5
vocab,10000,10000,10000
matrix,10000x10000,10000x10000,10000x10000
nonzero,1131356,2143234,4333662
density,1.13%,2.14%,4.33%
build_s,2.3,3.7,8.2
cos_doctor_physician_count,0.842,0.828,0.882
cos_doctor_physician_ppmi,0.107,0.099,0.139
cos_doctor_banana_ppmi,0.021,0.015,0.019
cos_cat_dog_ppmi,0.126,0.141,0.154


In [4]:
P5 = ppmi(mats[5])
for name, M in (("raw count, window=5", mats[5]), ("PPMI, window=5", P5)):
    print("==", name)
    for w in ("doctor","banana"):
        print(f"  {w:8s}", [(a, round(s,2)) for a, s in most_similar(w, M, vocab, 5)])

== raw count, window=5
  doctor   [('child', 0.94), ('job', 0.93), ('plan', 0.93), ('schedule', 0.93), ('or', 0.92)]
  banana   [('restaurant', 0.81), ('building', 0.81), ('leader', 0.81), ('major', 0.81), ('classic', 0.81)]
== PPMI, window=5
  doctor   [('patients', 0.19), ('patient', 0.18), ('surgery', 0.18), ('surgeon', 0.16), ('treatment', 0.16)]


  banana   [('tomato', 0.21), ('oats', 0.17), ('dried', 0.16), ('orange', 0.15), ('butter', 0.15)]


## Experiment 2 — Word2Vec
Training objective: **skip-gram + negative sampling** (`sg=1, negative=5`): cực đại hoá log σ(v_c·v_w) cho cặp (target, context) thật và log σ(−v_n·v_w) cho 5 negative lấy theo phân phối tần suất^0.75.

In [5]:
CFG = dict(vector_size=100, window=5, min_count=3, epochs=10, sg=1, negative=5, workers=8, seed=42)
def train(**kw):
    c = {**CFG, **kw}; t = time.time()
    m = Word2Vec(sentences, **c); m.train_time = time.time() - t
    return m
base = train()
print("corpus: C4 8k docs |", len(sentences), "sentences |", sum(map(len,sentences)), "tokens")
print("vocabulary:", len(base.wv), "| dim:", base.vector_size, "| window:", base.window,
      "| min_count: 3 | epochs: 10 | objective: skip-gram + negative sampling (neg=5)")
print("train time: %.1fs" % base.train_time)

corpus: C4 8k docs | 165900 sentences | 2875396 tokens
vocabulary: 33777 | dim: 100 | window: 5 | min_count: 3 | epochs: 10 | objective: skip-gram + negative sampling (neg=5)
train time: 37.8s


In [6]:
inspect_words = ["doctor","hospital","patient","disease","computer","football","banana"]
for w in inspect_words:
    top = base.wv.most_similar(w, topn=5)
    print(f"{w:9s}", [(a, round(s,2)) for a, s in top])
    log("exp2_w2v", "base(d100,w5,sg)", f"top5_{w}", " ".join(a for a,_ in top))

doctor    [('dentist', 0.75), ('physician', 0.72), ('ophthalmologist', 0.7), ('surgeon', 0.69), ('dermatologist', 0.67)]
hospital  [('chongqing', 0.64), ('obstetrics', 0.63), ('icu', 0.63), ('chiropractic', 0.63), ('cardiologist', 0.63)]
patient   [('doctor', 0.64), ('client', 0.61), ('initiating', 0.61), ('hygiene', 0.6), ('glossopharyngeal', 0.6)]
disease   [('chronic', 0.75), ('autoimmune', 0.72), ('syndrome', 0.72), ('artery', 0.72), ('diseases', 0.72)]
computer  [('emachines', 0.67), ('defraggler', 0.65), ('folders', 0.64), ('software', 0.64), ('desktop', 0.64)]
football  [('replica', 0.9), ('shirt', 0.83), ('shirts', 0.82), ('shirtr', 0.76), ('thcheap', 0.75)]
banana    [('cereals', 0.71), ('diced', 0.7), ('fennel', 0.7), ('confectioners', 0.7), ('pulp', 0.7)]


In [7]:
# Evidence từ corpus: các từ gần "doctor" có thực sự xuất hiện chung context không?
def cooc_count(a, b, win=5):
    n = 0
    for s in sentences:
        if a in s and b in s:
            ia = [i for i,x in enumerate(s) if x==a]; ib = [i for i,x in enumerate(s) if x==b]
            n += any(abs(i-j) <= win for i in ia for j in ib)
    return n
for b in ["physician","nurse","hospital","patient","disease","banana"]:
    same = sum(1 for s in sentences if 'doctor' in s and b in s)
    print(f"doctor ~ {b:10s} cùng câu: {same:4d} | trong 5 token: {cooc_count('doctor', b):4d} | cos = {base.wv.similarity('doctor', b):.3f}")

doctor ~ physician  cùng câu:    0 | trong 5 token:    0 | cos = 0.716


doctor ~ nurse      cùng câu:    2 | trong 5 token:    2 | cos = 0.619


doctor ~ hospital   cùng câu:    5 | trong 5 token:    1 | cos = 0.558
doctor ~ patient    cùng câu:   11 | trong 5 token:    5 | cos = 0.638


doctor ~ disease    cùng câu:    2 | trong 5 token:    0 | cos = 0.401


doctor ~ banana     cùng câu:    0 | trong 5 token:    0 | cos = -0.082


### Giải thích Top-5 (evidence từ corpus 8k docs, đếm số câu chứa từ)
Không thể nói "vì model học được semantic similarity". Dưới đây là lý do cụ thể:
- **doctor → dentist, physician, ophthalmologist, surgeon, dermatologist:** đều là danh từ chỉ nghề y, dùng thay cho nhau trong các câu kiểu "see a/your ___" ("my doctor", "the surgeon"...). Evidence: doctor và physician **0 câu chung** nhưng cos = 0.716; doctor đi cùng patient (11 câu), appointment (5), surgeon (5). Nghĩa là gần nhau nhờ *context chung*, không phải nhờ đứng cạnh nhau.
- **hospital → chongqing, obstetrics, icu, chiropractic, cardiologist:** là tên riêng/khoa/chuyên gia gắn với bệnh viện (bài viết giới thiệu địa điểm, danh sách khoa như "university college hospital ... obstetric"). `chongqing` là nhiễu do vài trang liệt kê bệnh viện ở địa phương đó. Evidence: hospital xuất hiện 273 câu, cardiologist đi chung 3 câu.
- **patient → doctor, client, initiating, hygiene, glossopharyngeal:** doctor (11 câu chung) là đối tượng tương tác; client là từ thay thế vai "người nhận dịch vụ"; glossopharyngeal/initiating là từ hiếm (ít dữ liệu nên vector thiếu ổn định).
- **disease → chronic, autoimmune, syndrome, artery, diseases:** cùng nhóm từ mô tả bệnh (tính từ/danh từ y học đi sát disease: "chronic disease", "artery disease").
- **computer → emachines, defraggler, folders, software, desktop:** các trang hỗ trợ kỹ thuật/hướng dẫn máy tính. software đi cùng 29 câu, desktop 4.
- **football → replica, shirt, shirts, shirtr, thcheap:** corpus có nhiều trang bán áo bóng đá: "replica" (69 câu) và "shirt" (66 câu) chung câu với football, trong khi "soccer" chỉ 13 câu. Đây là **domain bias** của web crawl.
- **banana → cereals, diced, fennel, confectioners, pulp:** từ liên quan nấu ăn/công thức; nhưng banana chỉ có 31 câu nên neighbor kém ổn định (cereals, fennel không đi chung câu nào, chỉ giống nhau về context).

### CBOW vs Skip-gram

In [8]:
cbow = train(sg=0)
for name, m in (("skip-gram", base), ("CBOW", cbow)):
    print(f"{name:9s} time={m.train_time:5.1f}s | doctor→", [a for a,_ in m.wv.most_similar('doctor', topn=5)])
    log("exp2_w2v", name, "train_time_s", round(m.train_time,1))

skip-gram time= 37.8s | doctor→ ['dentist', 'physician', 'ophthalmologist', 'surgeon', 'dermatologist']
CBOW      time= 16.4s | doctor→ ['physician', 'surgeon', 'child', 'mediator', 'patient']


## Experiment 3 — Context window (2 / 5 / 10)

In [9]:
pairs3 = [("doctor","physician"),("doctor","hospital"),("cat","dog"),("doctor","nurse"),("car","road")]
wmodels = {2: train(window=2), 5: base, 10: train(window=10)}
tab = pd.DataFrame({f"window {w}": [m.wv.similarity(a,b) for a,b in pairs3] for w, m in wmodels.items()},
                   index=[f"{a}–{b}" for a,b in pairs3]).round(3)
for w, m in wmodels.items():
    for (a,b) in pairs3: log("exp3_window", f"window={w}", f"sim_{a}-{b}", round(float(m.wv.similarity(a,b)),3))
    log("exp3_window", f"window={w}", "train_time_s", round(m.train_time,1))
display(tab)
for w, m in wmodels.items():
    print(f"window {w:2d} time={m.train_time:5.1f}s doctor→", [a for a,_ in m.wv.most_similar('doctor', topn=6)])

,window 2,window 5,window 10
doctor–physician,0.730,0.716,0.676
doctor–hospital,0.532,0.558,0.562
cat–dog,0.561,0.588,0.560
doctor–nurse,0.564,0.619,0.625
car–road,0.348,0.336,0.400


window  2 time= 22.9s doctor→ ['physician', 'dentist', 'surgeon', 'ophthalmologist', 'mediator', 'patient']
window  5 time= 37.8s doctor→ ['dentist', 'physician', 'ophthalmologist', 'surgeon', 'dermatologist', 'veterinarian']
window 10 time= 55.3s doctor→ ['dentist', 'ophthalmologist', 'surgeon', 'physician', 'genova', 'orthopedic']


### Window lớn hơn có luôn tốt hơn không? — Không
- doctor–physician **giảm** khi window tăng: 0.730 (w=2) → 0.716 (w=5) → 0.676 (w=10), vì context xa kéo thêm từ chỉ liên quan chủ đề thay vì từ thay thế được cho nhau.
- doctor–nurse và doctor–hospital **tăng nhẹ** (0.564 → 0.625; 0.532 → 0.562): quan hệ "cùng chủ đề" cần context rộng.
- car–road và cat–dog thay đổi không đơn điệu → hiệu ứng phụ thuộc loại quan hệ.
- Chi phí tăng: 22.9s → 37.8s → 55.3s. Window nhỏ cho similarity kiểu **đồng loại/cú pháp**, window lớn cho similarity kiểu **chủ đề**; với top-6 của doctor, window 10 bắt đầu xuất hiện nhiễu (genova, restitution...).

## Evaluation datasets (dùng chung cho Experiment 4 và mục Evaluation)
Word-similarity: điểm 0–10 do mình tự gán theo trực giác (không phải benchmark chuẩn như WordSim-353/SimLex-999, nên chỉ là kiểm tra thô). Đo bằng Spearman ρ.

In [10]:
SIM = [("doctor","physician",9.5),("car","automobile",9.5),("king","queen",8.0),("cat","dog",7.5),
       ("doctor","nurse",8.0),("doctor","hospital",7.0),("football","soccer",9.0),("computer","software",7.5),
       ("man","woman",7.5),("big","large",9.0),("happy","glad",8.5),("car","truck",8.0),
       ("doctor","disease",5.0),("money","bank",6.5),("football","stadium",6.0),("cat","banana",1.0),
       ("computer","banana",0.5),("doctor","banana",0.5),("car","football",1.0),("king","computer",0.5)]
def sim_eval(m):
    ok = [(a,b,h) for a,b,h in SIM if a in m.wv and b in m.wv]
    sims = [m.wv.similarity(a,b) for a,b,_ in ok]
    return spearmanr(sims, [h for *_,h in ok]).correlation, ok, sims
ANA = [("man","king","woman","queen"),("king","man","queen","woman"),("boy","girl","brother","sister"),
       ("paris","france","london","england"),("london","england","paris","france"),("good","better","bad","worse"),
       ("big","bigger","small","smaller"),("walk","walking","swim","swimming"),("he","his","she","her"),
       ("brother","sister","son","daughter"),("father","mother","uncle","aunt"),("tokyo","japan","beijing","china"),
       ("france","french","germany","german"),("quick","quickly","slow","slowly"),("great","greater","high","higher")]
# a : b :: c : d  -> b - a + c ≈ d
def ana_eval(m, topn=1):
    hit = n = 0
    for a,b,c,d in ANA:
        if all(x in m.wv for x in (a,b,c,d)):
            n += 1
            hit += d in [w for w,_ in m.wv.most_similar(positive=[b,c], negative=[a], topn=topn)]
    return hit / max(n,1), n
CATS = {"medical":"doctor nurse hospital patient disease surgery therapy clinic medicine cancer".split(),
        "sports":"football soccer basketball tennis baseball player coach stadium league tournament".split(),
        "food":"banana apple bread cheese chicken pizza coffee sugar rice butter".split(),
        "vehicle":"car truck bus train bicycle airplane motorcycle engine driver highway".split(),
        "tech":"computer software internet laptop server website database network keyboard programming".split()}
def downstream(m):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    X = np.array([m.wv[w] for ws in CATS.values() for w in ws if w in m.wv])
    y = [c for c, ws in CATS.items() for w in ws if w in m.wv]
    return cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5).mean()
rho, *_ = sim_eval(base); acc, n = ana_eval(base)
print(f"base: Spearman={rho:.3f} | analogy top-1 {acc:.2f} (n={n}) | word-category CV acc={downstream(base):.2f}")

base: Spearman=0.813 | analogy top-1 0.53 (n=15) | word-category CV acc=0.98


## Experiment 4 — Embedding dimension (50 / 100 / 300)

In [11]:
import os, tempfile
dmodels = {50: train(vector_size=50), 100: base, 300: train(vector_size=300)}
rows = []
for d, m in dmodels.items():
    rho, *_ = sim_eval(m); a1, n = ana_eval(m, 1); a5, _ = ana_eval(m, 5)
    p = os.path.join(tempfile.gettempdir(), f"w2v_{d}.kv"); m.wv.save(p)
    size = sum(os.path.getsize(os.path.join(tempfile.gettempdir(), f)) for f in os.listdir(tempfile.gettempdir()) if f.startswith(f"w2v_{d}.kv"))
    rows.append(dict(dim=d, train_s=round(m.train_time,1), size_MB=round(size/1e6,1), spearman=round(rho,3),
                     analogy_top1=round(a1,2), analogy_top5=round(a5,2), category_acc=round(downstream(m),2),
                     doctor_physician=round(float(m.wv.similarity("doctor","physician")),3)))
    for k, v in rows[-1].items():
        if k != "dim": log("exp4_dim", f"dim={d}", k, v)
display(pd.DataFrame(rows).set_index("dim"))

,train_s,size_MB,spearman,analogy_top1,analogy_top5,category_acc,doctor_physician
dim,,,,,,,
50,41.6,7.9,0.774,0.53,0.73,0.98,0.814
100,37.8,14.7,0.813,0.53,0.73,0.98,0.716
300,86.7,41.7,0.836,0.47,0.67,0.96,0.570


### Trade-off dimension ↔ capacity ↔ data ↔ computation
- Spearman tăng 0.774 → 0.813 → 0.836 nhưng chỉ trên 20 cặp (chênh lệch nhỏ, có thể do ngẫu nhiên); analogy top-1 và category accuracy **không tăng** (0.53/0.53/0.47; 0.98/0.98/0.96).
- Chi phí tăng rõ: dim 300 chậm gấp ~2.3 lần dim 100 (86.7s vs 37.8s) và nặng ~2.8 lần (41.7 vs 14.7 MB).
- Kết luận: với corpus ~2.9M token, dim 100 là điểm cân bằng; tăng dim chỉ có ích khi có thêm dữ liệu để học đủ các chiều.

## Evaluation — Word similarity (quantitative)

In [12]:
rho, ok, sims = sim_eval(base)
ev = pd.DataFrame([(a,b,h,round(float(s),3)) for (a,b,h),s in zip(ok,sims)], columns=["w1","w2","human","model_cos"])
ev["rank_human"] = ev.human.rank(ascending=False); ev["rank_model"] = ev.model_cos.rank(ascending=False)
ev["rank_gap"] = (ev.rank_human - ev.rank_model).abs()
display(ev.sort_values("model_cos", ascending=False).reset_index(drop=True))
print("Spearman rho (model vs intuition) = %.3f" % rho)
log("eval_sim", "base", "spearman", round(rho,3))
print("\nBất ngờ nhất (chênh hạng lớn):"); display(ev.sort_values("rank_gap", ascending=False).head(5)[["w1","w2","human","model_cos","rank_gap"]])

,w1,w2,human,model_cos,rank_human,rank_model,rank_gap
0,doctor,physician,9.5,0.716,1.5,1.0,0.5
1,happy,glad,8.5,0.714,5.0,2.0,3.0
2,car,truck,8.0,0.696,7.0,3.0,4.0
3,computer,software,7.5,0.643,10.0,4.0,6.0
4,doctor,nurse,8.0,0.619,7.0,5.0,2.0
5,king,queen,8.0,0.611,7.0,6.0,1.0
6,man,woman,7.5,0.608,10.0,7.0,3.0
7,car,automobile,9.5,0.592,1.5,8.0,6.5
8,cat,dog,7.5,0.588,10.0,9.0,1.0
9,football,soccer,9.0,0.575,3.5,10.0,6.5


Spearman rho (model vs intuition) = 0.813

Bất ngờ nhất (chênh hạng lớn):


,w1,w2,human,model_cos,rank_gap
9,big,large,9.0,0.515,8.5
1,car,automobile,9.5,0.592,6.5
6,football,soccer,9.0,0.575,6.5
7,computer,software,7.5,0.643,6.0
11,car,truck,8.0,0.696,4.0


## Evaluation — Word analogy
Đây là *pattern trong không gian vector*, không phải bằng chứng embedding “hiểu” quan hệ như con người.

In [13]:
print("king − man + woman →", [(w, round(s,3)) for w,s in base.wv.most_similar(positive=["king","woman"], negative=["man"], topn=5)])
res = []
for a,b,c,d in ANA:
    if all(x in base.wv for x in (a,b,c,d)):
        top = [w for w,_ in base.wv.most_similar(positive=[b,c], negative=[a], topn=5)]
        res.append((f"{a}:{b} :: {c}:?", d, top[0], d == top[0], d in top))
df = pd.DataFrame(res, columns=["query","expected","top1","hit@1","hit@5"]); display(df)
print("analogy hit@1 = %.2f, hit@5 = %.2f (n=%d)" % (df["hit@1"].mean(), df["hit@5"].mean(), len(df)))
log("eval_analogy", "base", "hit@1", round(df["hit@1"].mean(),2)); log("eval_analogy", "base", "hit@5", round(df["hit@5"].mean(),2))

king − man + woman → [('queen', 0.554), ('cal', 0.536), ('headboard', 0.456), ('lizzy', 0.447), ('bed', 0.443)]


,query,expected,top1,hit@1,hit@5
0,man:king :: woman:?,queen,queen,True,True
1,king:man :: queen:?,woman,woman,True,True
2,boy:girl :: brother:?,sister,mother,False,True
3,paris:france :: london:?,england,denmark,False,True
4,london:england :: paris:?,france,tulsa,False,False
5,good:better :: bad:?,worse,worse,True,True
6,big:bigger :: small:?,smaller,smaller,True,True
7,walk:walking :: swim:?,swimming,auntie,False,False
8,he:his :: she:?,her,her,True,True
9,brother:sister :: son:?,daughter,daughter,True,True


analogy hit@1 = 0.53, hit@5 = 0.73 (n=15)


In [14]:
# Bài tập tính analogy (đề bài): vector giả định
king, man, woman = np.array([8,2,7]), np.array([5,1,5]), np.array([5,3,5])
v = king - man + woman
print("king − man + woman =", v, "| cos(v, king) = %.3f | cos(v, woman) = %.3f" % (cosine_similarity(v,king), cosine_similarity(v,woman)))

king − man + woman = [8 4 7] | cos(v, king) = 0.985 | cos(v, woman) = 0.997


## Application — Semantic search
So sánh với LAB 01 (TF-IDF): (a) TF-IDF thuần, (b) query expansion bằng từ gần trong Word2Vec rồi TF-IDF, (c) document = trung bình word vector có trọng số IDF.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
rng = random.Random(1)
corpus_docs = [d for d in docs if 60 <= len(d.split()) <= 400]
pool = rng.sample(corpus_docs, min(1500, len(corpus_docs)))
tf = TfidfVectorizer(stop_words="english", min_df=2); T = tf.fit_transform(pool)
idf = dict(zip(tf.get_feature_names_out(), tf.idf_))
wv = base.wv
def doc_vec(text):
    ws = [w for w in tokenize(text) if w in wv and w in idf]
    if not ws: return np.zeros(wv.vector_size)
    v = np.average([wv[w] for w in ws], axis=0, weights=[idf[w] for w in ws]); return v / (np.linalg.norm(v) or 1)
D = np.array([doc_vec(d) for d in pool])
def expand(query, k=4):
    out = list(tokenize(query))
    for w in tokenize(query):
        if w in wv: out += [a for a,_ in wv.most_similar(w, topn=k)]
    return out
def search_tfidf(q, k=5):
    s = (T @ tf.transform([q]).T).toarray().ravel(); return np.argsort(-s)[:k], s
def search_expanded(q, k=5): return search_tfidf(" ".join(expand(q)), k)
def search_emb(q, k=5):
    s = D @ doc_vec(q); return np.argsort(-s)[:k], s
snip = lambda i: " ".join(pool[i].split()[:22]) + " …"
q = "medical treatment"
print("Query:", q, "| mở rộng:", expand(q))
for name, fn in (("TF-IDF", search_tfidf), ("TF-IDF + expansion", search_expanded), ("Embedding avg", search_emb)):
    print("\n==", name)
    for i in fn(q)[0][:3]: print("  -", snip(i))

Query: medical treatment | mở rộng: ['medical', 'treatment', 'malpractice', 'veterinary', 'counseling', 'pathology', 'surgical', 'cataract', 'enhancement', 'dermatologist']

== TF-IDF
  - Stefi has degrees in medical history (UC Berkeley) and international public health (UCLA). In 2010 she co-founded the Sustainable Healthcare Education network …
  - Cosmetic surgery has become increasingly popular in the UK over the last ten years. Social media and cosmetically-enhanced reality TV stars, on …
  - MONDAY, Nov. 2, 2009 (HealthDay News) -- The length of time a patient has to wait between lung cancer diagnosis and treatment …

== TF-IDF + expansion
  - How to locate advice you can trust. Credit counseling is a fine idea. The government thinks so highly of it that the …
  - Stefi has degrees in medical history (UC Berkeley) and international public health (UCLA). In 2010 she co-founded the Sustainable Healthcare Education network …
  - The Medical Center, Navicent Health (MCNH) has receive

In [16]:
# Đánh giá định lượng thô: tài liệu "đúng chủ đề" = chứa >= 3 từ trong bộ từ khóa y tế rộng
MED = set("medical doctor hospital patient patients disease therapy clinical treatment health surgery physician nurse medicine drug cancer symptoms diagnosis".split())
rel = np.array([len(MED & set(tokenize(d))) >= 3 for d in pool]); print("relevant docs:", rel.sum(), "/", len(pool))
for q in ["medical treatment", "doctor hospital", "cancer therapy"]:
    for name, fn in (("TF-IDF", search_tfidf), ("TF-IDF+expansion", search_expanded), ("Embedding avg", search_emb)):
        idx = fn(q, 20)[0]; p = rel[idx].mean()
        print(f"{q:18s} {name:17s} P@20 = {p:.2f}"); log("application_search", f"{q}|{name}", "P@20", round(float(p),2))

relevant docs: 45 / 1500


medical treatment  TF-IDF            P@20 = 0.80
medical treatment  TF-IDF+expansion  P@20 = 0.60
medical treatment  Embedding avg     P@20 = 0.90
doctor hospital    TF-IDF            P@20 = 0.65
doctor hospital    TF-IDF+expansion  P@20 = 0.65
doctor hospital    Embedding avg     P@20 = 0.80
cancer therapy     TF-IDF            P@20 = 0.50
cancer therapy     TF-IDF+expansion  P@20 = 0.45
cancer therapy     Embedding avg     P@20 = 0.80


## Error analysis & polysemy

In [17]:
for a, b in [("doctor","nurse"),("doctor","hospital"),("doctor","disease"),("doctor","physician"),("car","automobile"),("cat","banana")]:
    nb_list = [w for w,_ in base.wv.most_similar(a, topn=len(base.wv))]
    print(f"{a:7s}~{b:10s} cos = {base.wv.similarity(a,b):.3f} | rank của {b} trong neighbors({a}) = {nb_list.index(b)+1}")
for w in ("physician","automobile","doctor"): print(w, "freq:", base.wv.get_vecattr(w, "count"))
print("\nbank →", [(a, round(s,2)) for a,s in base.wv.most_similar("bank", topn=10)])
def ctx(w, k=4, seed=3):
    r = random.Random(seed); ss = [s for s in sentences if w in s]; return [" ".join(s[:18]) for s in r.sample(ss, k)]
print("\nVí dụ câu chứa 'bank':"); [print("  -", s) for s in ctx("bank", 6)]

doctor ~nurse      cos = 0.619 | rank của nurse trong neighbors(doctor) = 22
doctor ~hospital   cos = 0.558 | rank của hospital trong neighbors(doctor) = 134


doctor ~disease    cos = 0.401 | rank của disease trong neighbors(doctor) = 6281


doctor ~physician  cos = 0.716 | rank của physician trong neighbors(doctor) = 2


car    ~automobile cos = 0.592 | rank của automobile trong neighbors(car) = 24
cat    ~banana     cos = 0.094 | rank của banana trong neighbors(cat) = 32870
physician freq: 73
automobile freq: 15
doctor freq: 254

bank → [('credit', 0.69), ('citibank', 0.67), ('td', 0.66), ('ammadam', 0.65), ('banking', 0.65), ('intercompany', 0.65), ('procredit', 0.64), ('ifsc', 0.64), ('phoebus', 0.63), ('cheque', 0.63)]

Ví dụ câu chứa 'bank':


  - to satisfy typical generator loading requirements the load bank is normally sized between and of maximum generator capacity
  - keep in mind if your bank is closed most major banks have mobile apps that allow you to
  - for candidates having experience as an officer in commercial bank financial institution in public sector as also prior
  - the financial times provides some useful background on the struggle toward transparency at the vatican bank a timely
  - you can search for a particular branch name in the table which provides ifsc codes for all branches
  - to send a secured message you must first login to your td bank online profile


[None, None, None, None, None, None]

## Lưu kết quả

In [18]:
with open("results.csv", "w", newline="", encoding="utf8") as f:
    wr = csv.writer(f); wr.writerow(["experiment","config","metric","value"]); wr.writerows(RESULTS)
print(len(RESULTS), "rows → results.csv")

72 rows → results.csv
